In [0]:
# Auto Loader: CSV files to Bronze Table
# Note: This will run successfully only when CSV files are present in the source path

# Source and target configuration
source_path = "/Volumes/retail_oc/blob_volume/csv_volume"
schema_location = "/Volumes/retail_oc/blob_volume/csv_volume/_schema"
checkpoint_location = "/Volumes/retail_oc/blob_volume/csv_volume/_checkpoint"
target_table = "retail_oc.blob_bronze.transaction"

# Read CSV files using Auto Loader
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaLocation", schema_location)
  .option("cloudFiles.schemaEvolutionMode", "rescue")  # Handles schema changes gracefully
  .load(source_path)
)

# Write to bronze table with incremental processing
query = (df.writeStream
  .option("checkpointLocation", checkpoint_location)
  .option("mergeSchema", "true")
  .trigger(availableNow=True)  # Process all available files then stop
  .toTable(target_table)
)

# Display query status
print(f"Stream started. Writing to {target_table}")
query.awaitTermination()
print(f"Stream stopped. Data written to {target_table}")

In [0]:
%sql
SELECT count(*) FROM retail_oc.blob_bronze.transaction;